[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/halla-ai/deepnlp-2026/blob/main/notebooks/week-07.ipynb)

# 7주차 실습: 장문맥 처리와 효율적 추론

**목표.** 긴 한국어 문서(위키백과 「제주특별자치도」 고정판)로 세 가지를 해 본다.

1. **문맥이 길어지면 무엇이 비싸지는가**: 모델 설정에서 KV 캐시 크기를 계산하고, 프리필(입력 읽기)과 디코딩(한 토큰씩 쓰기) 시간을 직접 잰다
2. **위치 인코딩이 길이를 정하는 방식**: RoPE의 차원 쌍별 회전 주기를 계산해 학습 길이 밖이 왜 문제인지 본다
3. **위치별 정보 회수율**: 문서 곳곳에 가상의 문장을 숨기고 모델이 찾아내는지 잰다. **문서 길이를 바꿔 가며** 길이-정확도 곡선을 그린다

GPU는 필요 없다. 노트북이 실행 시점에 문서와 모델을 내려받는다. 문서는 위키백과 본문(CC BY-SA 4.0)이고, 숨기는 문장(안내센터 출입 번호)은 실험용으로 지어낸 것이다.

## 0. 준비

아래 셀을 실행해 필요한 라이브러리를 설치한다. GPU는 필요 없다.

In [ ]:
# 필요한 것 설치 (Colab에서 한 번만)
!pip -q install -U transformers
!pip -q install torch matplotlib

## 1. 먼저 그냥 실행해 보기

아래 셀들을 위에서부터 차례로 실행하세요. 아무것도 고치지 않아도 끝까지 돌아갑니다.

### 1-1. 문서 불러오기 - 위키백과 고정판

위키백과 「제주특별자치도」 문서의 **2026-09-11 판(판 번호 42464087)** 을 내려받는다. 판 번호를 고정했으므로 누가 언제 실행해도 본문은 사실상 같은 글이다(위키백과의 틀이 바뀌면 글자 수가 조금 달라질 수 있다). 표, 각주, 목차는 버리고 본문 문단만 쓴다.

In [ ]:
import re, html, json, urllib.request, urllib.parse

OLDID = 42464087  # 위키백과 「제주특별자치도」 2026-09-11 판 (고정)
url = "https://ko.wikipedia.org/w/api.php?" + urllib.parse.urlencode(
    {"action": "parse", "oldid": OLDID, "prop": "text", "format": "json", "formatversion": 2})
req = urllib.request.Request(url, headers={"User-Agent": "deepnlp-course/1.0 (yj.lee@chu.ac.kr)"})
raw = json.load(urllib.request.urlopen(req, timeout=60))["parse"]["text"]

def clean(p):
    p = re.sub(r"<sup.*?</sup>", "", p, flags=re.S)  # 각주 번호 제거
    p = re.sub(r"<[^>]+>", "", p)                     # 태그 제거
    return re.sub(r"\s+", " ", html.unescape(p)).strip()

paras = [clean(p) for p in re.findall(r"<p>(.*?)</p>", raw, flags=re.S)]
paras = [p for p in paras if len(p) > 40]
# 문장 단위로 자른다. 숨길 문장을 문장 사이에 끼우기 위해서다
sents = [s.strip() for p in paras for s in re.split(r"(?<=[.다])\s+", p) if s.strip()]
n_chars = sum(len(p) for p in paras)
print(f"문단 {len(paras)}개, 문장 {len(sents)}개, {n_chars:,}자")
print("첫 문장:", sents[0][:80], "...")

### 1-2. 모델 불러오기와 설정 읽기

4~6주차와 같은 Qwen3-0.6B-Base를 쓴다. 장문맥과 관련된 설정값(층 수, 헤드 수, 최대 위치, RoPE 기저)을 먼저 읽어 둔다. 아래 계산이 모두 이 값에서 나온다.

In [ ]:
import time, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
torch.manual_seed(0)

name = "Qwen/Qwen3-0.6B-Base"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32).eval()
cfg = model.config
rope_theta = cfg.rope_parameters["rope_theta"] if getattr(cfg, "rope_parameters", None) else cfg.rope_theta
n_params = sum(p.numel() for p in model.parameters())

print(f"층 수              {cfg.num_hidden_layers}")
print(f"쿼리 헤드 수        {cfg.num_attention_heads}")
print(f"KV 헤드 수          {cfg.num_key_value_heads}  (쿼리 헤드 {cfg.num_attention_heads // cfg.num_key_value_heads}개가 KV 헤드 1개를 함께 쓴다 = GQA)")
print(f"헤드 차원           {cfg.head_dim}")
print(f"최대 위치           {cfg.max_position_embeddings:,}")
print(f"RoPE 기저 b         {rope_theta:,.0f}")
print(f"파라미터 수         {n_params:,}")

doc_tokens = len(tok(" ".join(sents), add_special_tokens=False).input_ids)
print(f"\n문서 전체 {n_chars:,}자 = {doc_tokens:,}토큰 (토큰 하나에 평균 {n_chars / doc_tokens:.2f}자)")

### 1-3. KV 캐시 크기 계산 - 문맥 길이에 정비례

디코딩할 때 모델은 앞 토큰들의 Key와 Value를 층마다 저장해 둔다(KV 캐시). 크기는 다음과 같다.

`KV 캐시 = 2 (K와 V) x 층 수 x KV 헤드 수 x 헤드 차원 x 바이트 x 토큰 수`

모델을 돌리지 않고 설정값만으로 계산한다. 바이트는 배포에 흔히 쓰는 bf16(2바이트)으로 잡는다. 이 노트북은 CPU 속도 때문에 fp32(4바이트)로 돌리므로 실제 메모리는 2배다.

In [ ]:
BYTES = 2  # bf16
per_token = 2 * cfg.num_hidden_layers * cfg.num_key_value_heads * cfg.head_dim * BYTES
weights = n_params * BYTES
print(f"토큰당 KV 캐시: {per_token:,}바이트 = {per_token / 1024:.0f} KiB")
print(f"가중치 (bf16):  {weights / 2**30:.2f} GiB\n")
print(f"{'문맥 길이':>10} | {'KV 캐시':>10} | 가중치 대비")
for n in [1024, 4096, 8192, 16384, 32768]:
    kv = per_token * n
    print(f"{n:>10,} | {kv / 2**30:>6.2f} GiB | {kv / weights:>5.2f}배")
print(f"\nKV 캐시가 가중치와 같아지는 길이: 약 {weights / per_token:,.0f}토큰")
mha = 2 * cfg.num_hidden_layers * cfg.num_attention_heads * cfg.head_dim * BYTES
print(f"KV 헤드를 쿼리 헤드 수({cfg.num_attention_heads})만큼 두었다면(MHA) 토큰당 {mha / 1024:.0f} KiB, 지금의 {mha / per_token:.0f}배")

### 1-4. RoPE 주기 - 학습 길이 밖은 왜 문제인가

RoPE는 위치 i의 토큰을 차원 쌍마다 각도 i x θ_k 만큼 돌린다. 차원 쌍 k의 회전 속도는 θ_k = b^(-2(k-1)/d) 이고, 한 바퀴 도는 데 걸리는 토큰 수(주기)는 T_k = 2π x b^(2(k-1)/d) 다. 빠른 쌍은 몇 토큰마다 한 바퀴를 돌고, 느린 쌍은 최대 위치 안에서 한 바퀴도 못 돈다. 느린 쌍은 학습 때 본 각도 범위가 좁으므로, 학습 길이를 넘으면 처음 보는 각도가 나온다.

In [ ]:
import math
d = cfg.head_dim
T = [2 * math.pi * rope_theta ** (2 * (k - 1) / d) for k in range(1, d // 2 + 1)]
ML = cfg.max_position_embeddings
print(f"차원 쌍 {d // 2}개")
print(f"가장 빠른 쌍 (k=1):  {T[0]:.2f}토큰마다 한 바퀴")
print(f"가장 느린 쌍 (k={d // 2}): {T[-1]:,.0f}토큰마다 한 바퀴")
print(f"최대 위치 {ML:,} 안에서 한 바퀴도 못 도는 쌍: {sum(t > ML for t in T)}개")
print(f"가장 느린 쌍이 위치 {ML:,}까지 도는 각도: {math.degrees(2 * math.pi * ML / T[-1]):.1f}도")

# 위치 보간: 4배 긴 문맥(m)을 학습 길이(m_l) 안에 넣으려면 위치를 m_l / m 배로 줄인다
m = 4 * ML
print(f"\n위치 보간 예: 길이 {m:,}를 {ML:,} 안에 넣기 -> 위치 100,000은 {100_000 * ML / m:,.0f}로 바뀐다")
# 기저 늘리기: 가장 느린 쌍의 주기가 위치 보간과 같아지도록 b를 lambda배 한다
lam = (m / ML) ** (d / (d - 2))
T2 = [2 * math.pi * (lam * rope_theta) ** (2 * (k - 1) / d) for k in range(1, d // 2 + 1)]
print(f"기저 늘리기 예: lambda = {lam:.4f}, 가장 빠른 쌍 주기 {T[0]:.2f} -> {T2[0]:.2f} (그대로), 가장 느린 쌍 {T[-1]:,.0f} -> {T2[-1]:,.0f} ({T2[-1] / T[-1]:.2f}배)")

### 1-5. 프리필과 디코딩 시간 재기

추론은 두 단계다. **프리필**은 입력 전체를 한 번에 읽어 KV 캐시를 만든다. **디코딩**은 그 캐시를 읽으며 한 토큰씩 쓴다. 문서 앞부분을 길이별로 잘라 프리필 시간을 재고, 가장 긴 입력에서 디코딩 토큰당 시간을 잰다.

- 프리필은 `logits_to_keep=1`로 잰다. 다음 토큰을 고르는 데 필요한 마지막 자리의 확률만 계산하는 것으로, `generate`가 실제로 하는 방식과 같다
- 디코딩 토큰당 시간은 1토큰 생성 시간(첫 토큰까지 시간)과 20토큰 생성 시간의 차이를 19로 나눠 구한다

시간은 실행 환경마다 다르다. 절대값보다 **길이에 따라 늘어나는 모양**을 본다.

In [ ]:
full_ids = tok(" ".join(sents), return_tensors="pt").input_ids
PREFILL_LENGTHS = [250, 500, 1000, 2000]
prefill_sec = {}
with torch.no_grad():
    model(full_ids[:, :50], logits_to_keep=1)  # 첫 호출의 준비 시간을 측정에서 빼기 위한 예열
    for n in PREFILL_LENGTHS:
        t = time.time()
        model(full_ids[:, :n], logits_to_keep=1)  # 마지막 자리의 확률만 계산 (generate와 같은 방식)
        prefill_sec[n] = time.time() - t
        print(f"입력 {n:>5,}토큰 프리필: {prefill_sec[n]:.2f}초")

n = PREFILL_LENGTHS[-1]
NEW = 20
def gen_time(k):
    t = time.time()
    with torch.no_grad():
        model.generate(full_ids[:, :n], max_new_tokens=k, min_new_tokens=k, do_sample=False, pad_token_id=tok.eos_token_id)
    return time.time() - t
ttft = gen_time(1)       # 첫 토큰까지 시간: 프리필 + 첫 토큰
total = gen_time(NEW)
decode_per_tok = (total - ttft) / (NEW - 1)
print(f"\n입력 {n:,}토큰 뒤 1토큰 생성(첫 토큰까지 시간): {ttft:.2f}초, {NEW}토큰 생성: {total:.2f}초")
print(f"  이후 토큰당 약 {decode_per_tok:.3f}초 = ({total:.2f} - {ttft:.2f}) / {NEW - 1}")

### 1-6. 숨긴 문장 찾기 - 측정 도구

문서 사이사이에 **가상의 안내센터 출입 번호 문장 열 개**를 끼운다(성산, 한림, 애월 ... 안내센터마다 번호가 다르다). 그중 하나를 골라 번호를 묻는다. 찾을 문장은 정해진 깊이(0% = 맨 앞, 100% = 맨 뒤)에 놓고, 나머지 아홉 문장은 문서 전체에 고르게 흩어 둔다. 비슷한 문장이 여럿이라 **정확히 그 안내센터**의 번호를 골라야 맞는다.

- 문서 길이 L은 위키백과 본문의 토큰 수다(숨긴 문장과 질문은 빼고 센 값)
- 답은 탐욕 디코딩으로 최대 6토큰을 생성하고, 정답 번호가 들어 있으면 회수 성공으로 센다
- 답의 첫머리("{안내센터} 안내센터의 오늘 임시 출입 번호는")는 프롬프트 끝에 미리 적어 둔다. 작은 바탕 모델은 질문을 되풀이하는 습관이 있어서다. 첫머리를 빼면 무슨 일이 생기는지는 3에서 직접 잰다

In [ ]:
import random

sent_tok = [len(tok(" " + s, add_special_tokens=False).input_ids) for s in sents]

def haystack(n_tokens):
    """문서 앞에서부터 문장을 모아 n_tokens를 넘지 않게 자른다"""
    out, total = [], 0
    for s, k in zip(sents, sent_tok):
        if total + k > n_tokens:
            break
        out.append(s)
        total += k
    return out

PLACES = ["성산", "한림", "애월", "구좌", "대정", "표선", "남원", "조천", "안덕", "한경"]
CODES = {p: str(c) for p, c in zip(PLACES, random.Random(7).sample(range(1000, 10000), len(PLACES)))}
TARGETS = ["애월", "표선"]          # 번호를 물을 안내센터 (깊이마다 둘 다 묻는다)
DEPTHS = [0.0, 0.25, 0.5, 0.75, 1.0]

def fact(place):
    return f"{place} 안내센터의 오늘 임시 출입 번호는 {CODES[place]}이다."

def build_prompt(n_tokens, depth, target, answer_prefix=True):
    hs = haystack(n_tokens)
    others = [p for p in PLACES if p != target]
    items = [(depth, fact(target))] + [((k + 0.5) / len(others), fact(p)) for k, p in enumerate(others)]
    doc = list(hs)
    for pos, _, s in sorted(((round(f * len(hs)), i, s) for i, (f, s) in enumerate(items)), reverse=True):
        doc.insert(pos, s)
    q = f"질문: {target} 안내센터의 오늘 임시 출입 번호는 무엇인가?"
    tail = f"\n답: {target} 안내센터의 오늘 임시 출입 번호는" if answer_prefix else "\n답:"
    return "다음 문서를 읽고 질문에 답하시오.\n\n문서:\n" + " ".join(doc) + f"\n\n{q}" + tail

def ask(prompt, max_new_tokens=6):
    ids = tok(prompt, return_tensors="pt").input_ids
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True), ids.shape[1]

WHO = {v: k for k, v in CODES.items()}

def run_grid(lengths, answer_prefix=True, max_new_tokens=6, show_miss=True):
    grid = {}
    for L in lengths:
        t0 = time.time()
        for depth in DEPTHS:
            hits = 0
            for tg in TARGETS:
                ans, n_in = ask(build_prompt(L, depth, tg, answer_prefix), max_new_tokens)
                ok = CODES[tg] in ans
                hits += ok
                if not ok and show_miss:
                    m = re.search(r"\d{4}", ans)
                    who = WHO.get(m.group(0), "없는 번호") if m else "번호 없음"
                    print(f"  놓침 L={L} 깊이 {depth:.0%} {tg}: 답 {ans.strip()!r} -> {who}")
            grid[(L, depth)] = hits / len(TARGETS)
        row = "  ".join(f"{d:>4.0%}:{grid[(L, d)]:.1f}" for d in DEPTHS)
        print(f"L={L:>5,} (입력 약 {n_in:,}토큰, {time.time() - t0:.0f}초) | {row}")
    return grid

# 동작 확인: 짧은 문서에서 한 건
print("숨긴 번호:", CODES)
ans, n_in = ask(build_prompt(250, 0.5, "애월"))
print(f"\n입력 {n_in}토큰, 깊이 50%에 애월 문장 -> 답: {ans.strip()!r} (정답 {CODES['애월']})")

## 2. 한 지점만 바꿔 보기 - 문서 길이

아래 셀의 `# TODO`로 표시된 **한 곳만** 바꾸고 다시 실행하세요.

> 규칙: `LENGTHS`에 문서 길이(토큰 수)를 넣는다. 기본값은 짧은 길이 셋이다. 길이를 늘려 넣고(예: 2000, 4000) 길이-정확도 곡선이 어떻게 바뀌는지 기록한다. 문서 전체가 약 7,800토큰이므로 7,000을 넘기지 않는다. 길이가 늘면 시간도 는다. 2,000토큰을 넘으면 길이보다 빠르게 늘 수 있다(실행 뒤 행마다 찍히는 시간을 비교해 보자). 이것이 학습활동 [구현]의 기록이다.

In [ ]:
# TODO: 문서 길이(토큰 수)를 바꾼다. 예: [500, 1000, 2000, 4000]  (7,000 이하)
LENGTHS = [250, 500, 1000]

# 아래는 그대로 둡니다
LENGTHS = sorted(set(LENGTHS))
grid = run_grid(LENGTHS)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
plt.rcParams["axes.unicode_minus"] = False

acc = [np.mean([grid[(L, d)] for d in DEPTHS]) for L in LENGTHS]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(LENGTHS, acc, "o-")
ax1.set_xscale("log"); ax1.minorticks_off(); ax1.set_xticks(LENGTHS); ax1.set_xticklabels([str(L) for L in LENGTHS])
ax1.set_ylim(-0.05, 1.05); ax1.set_xlabel("document length (tokens)"); ax1.set_ylabel("recall (mean over depths)")
ax1.set_title("length - accuracy")
M = np.array([[grid[(L, d)] for d in DEPTHS] for L in LENGTHS])
im = ax2.imshow(M, vmin=0, vmax=1, cmap="Greens", aspect="auto")
ax2.set_xticks(range(len(DEPTHS))); ax2.set_xticklabels([f"{d:.0%}" for d in DEPTHS])
ax2.set_yticks(range(len(LENGTHS))); ax2.set_yticklabels([str(L) for L in LENGTHS])
ax2.set_xlabel("depth of the target sentence"); ax2.set_ylabel("document length")
for i in range(len(LENGTHS)):
    for j in range(len(DEPTHS)):
        ax2.text(j, i, f"{M[i, j]:.1f}", ha="center", va="center", color="white" if M[i, j] > 0.6 else "black")
ax2.set_title("recall by length and depth")
plt.tight_layout(); plt.show()
for L, a in zip(LENGTHS, acc):
    print(f"L={L:>5,}: 평균 회수율 {a:.2f}")

## 3. 같은 모델, 다른 채점 - 답의 첫머리를 빼면

지금까지는 프롬프트 끝에 답의 첫머리("{안내센터} 안내센터의 오늘 임시 출입 번호는")를 미리 적어 두고 6토큰만 생성했다. 이번에는 첫머리를 빼고 `답:`에서 끝낸다. 생성 길이는 8토큰과 24토큰 두 가지로 잰다. 문서와 숨긴 문장, 모델은 그대로다.

결과를 보기 전에 예측해 보자. 모델이 번호를 "아는지"는 바뀌지 않았다. 그런데 회수율도 그대로일까?

In [ ]:
L_fmt = 500
print("[답 첫머리 없음, 8토큰 생성]")
g8 = run_grid([L_fmt], answer_prefix=False, max_new_tokens=8, show_miss=False)
print("\n[답 첫머리 없음, 24토큰 생성]")
g24 = run_grid([L_fmt], answer_prefix=False, max_new_tokens=24, show_miss=False)
print("\n깊이 50% 애월의 답 비교")
for prefix, n_new in [(True, 6), (False, 8), (False, 24)]:
    ans, _ = ask(build_prompt(L_fmt, 0.5, "애월", answer_prefix=prefix), n_new)
    print(f"  첫머리 {'있음' if prefix else '없음'}, {n_new:>2}토큰: {ans.strip()!r}")

## 4. 확인 질문

1. 1-3에서 KV 캐시가 가중치와 같아지는 길이는 몇 토큰이었나요? 32,768토큰이면 가중치의 몇 배인가요?
2. 1-4에서 최대 위치 안에서 한 바퀴도 못 도는 차원 쌍은 몇 개였나요? 학습 길이를 넘는 위치에서 이 쌍들에 무슨 일이 생기나요?
3. 1-5에서 입력이 2배가 될 때 프리필 시간은 몇 배가 되었나요? 2,000토큰 입력 뒤 디코딩 토큰당 시간은 프리필 시간의 몇 분의 1이었나요? 짧은 답을 받을 때 기다리는 시간의 대부분은 어느 단계인가요?
4. 2에서 길이를 늘렸을 때 회수율은 어떻게 바뀌었나요? 떨어졌다면 어느 깊이에서 먼저 떨어졌나요? 놓친 답은 어떤 번호였나요(다른 안내센터의 번호인가, 없는 번호인가)?
5. 3에서 답의 첫머리를 빼고 8토큰만 생성했을 때 회수율은 얼마였나요? 24토큰에서는요? 모델이 번호를 "알았는지"와 회수율 숫자는 어떤 관계인가요?
6. 이 실험에서 회수율이 1.0이라면 "모델이 문서 전체를 이해했다"고 말할 수 있나요? 이 측정이 말해 주는 것과 말해 주지 않는 것을 한 문장씩 써 보세요.

답은 아래 셀에 글로 적으면 됩니다. 코드가 아니어도 됩니다.

*(여기에 답을 적으세요)*

## 5. 제출

1. 상단 메뉴 **파일 > .ipynb 다운로드** 로 이 노트북을 내려받습니다
2. [저장소](https://github.com/halla-ai/deepnlp-2026)의 `assignments/week-07/<내 학번>/` 에 업로드합니다
3. Pull Request를 엽니다

자세한 방법은 강의 사이트의 **과제 제출** 문서에 있습니다.

---

**막혔나요?** 오류 메시지의 마지막 줄을 먼저 읽어 보세요. 그래도 안 되면 AI Professor 튜터에게 묻고, 그래도 막히면 저장소 Issues에 남기세요.